# Обучение нейронной сети AlexNet на датасете CIFAR-100

## Описание модели: AlexNet
**AlexNet** — это одна из первых глубоких сверточных нейронных сетей, которая произвела революцию в области компьютерного зрения. Модель была предложена в 2012 году Алексеем Крижевским, Ильей Сутскевером и Джеффри Хинтоном и выиграла конкурс ImageNet Large Scale Visual Recognition Challenge (ILSVRC). Основные элементы модели:
- **Сверточные слои (Convolutional Layers)**:
  - Извлекают пространственные и текстурные признаки из изображений.
  - Активация ReLU ускоряет процесс обучения и добавляет нелинейность.
- **Слои пулинга (Pooling Layers)**:
  - Уменьшают размерность признаков и помогают избежать переобучения.
- **Полносвязанные слои (Fully Connected Layers)**:
  - Используются для окончательной классификации изображения на один из классов.
- **Dropout**:
  - Регуляризация для уменьшения риска переобучения.

### Адаптация модели для CIFAR-100
Оригинальная AlexNet была разработана для работы с большими изображениями (224x224), такими как в датасете ImageNet. В данном проекте модель адаптирована под размер изображений 32x32:
- Уменьшены размеры сверток и пулов.
- Полносвязанные слои пересчитаны с учетом меньших входных данных.

---

## Описание датасета: CIFAR-100
**CIFAR-100** (Canadian Institute For Advanced Research) — это популярный датасет для задач классификации изображений. Его особенности:
- Состоит из **60,000 изображений размером 32x32 пикселя**.
- Изображения распределены по **100 классам**, таких как "млекопитающие", "растения", "транспорт".
- Разделение данных:
  - **Обучающая выборка**: 50,000 изображений.
  - **Тестовая выборка**: 10,000 изображений.
- Каждый класс содержит 600 изображений, что обеспечивает равномерное распределение данных.

---

## Используемые библиотеки
Для реализации проекта используются современные и популярные инструменты машинного обучения:
1. **PyTorch**:
   - Гибкая и мощная библиотека для работы с нейронными сетями.
   - Обеспечивает выполнение операций с тензорами на GPU.
2. **PyTorch Lightning**:
   - Упрощает процесс обучения и структурирует код.
   - Автоматически логирует метрики, облегчает работу с несколькими устройствами (GPU/TPU).
3. **Torchvision**:
   - Библиотека, предоставляющая предобработанные датасеты, преобразования изображений и предобученные модели.
4. **Torchmetrics**:
   - Позволяет вычислять метрики классификации (точность, Precision, Recall, F1 Score).
5. **Matplotlib**:
   - Используется для построения графиков и визуализации результатов.
6. **Google Colab**:
   - Облачная среда для выполнения Python-кода с поддержкой GPU.

---

## План работы
1. **Подготовка данных**:
   - Загрузка датасета CIFAR-100 из библиотеки Torchvision.
   - Разделение данных на обучающую, валидационную и тестовую выборки.
   - Преобразование изображений с использованием `transforms` (нормализация и приведение к тензору).
2. **Реализация модели**:
   - Построение сверточной нейронной сети AlexNet с адаптацией под размер 32x32.
   - Определение функции потерь (CrossEntropyLoss).
   - Настройка оптимизатора (Adam).
3. **Обучение модели**:
   - Инициализация тренировочного процесса на 10 эпохах.
   - Продолжение обучения еще на 50 эпох.
   - Логирование метрик (Loss, Accuracy, Precision, Recall, F1 Score).
4. **Тестирование модели**:
   - Оценка производительности модели на тестовых данных.
   - Расчет метрик и их интерпретация.

## Импорт необходимых библиотек

На данном этапе мы подключаем все необходимые библиотеки и инструменты, которые будут использоваться в проекте:

1. **PyTorch**:
   - Основной фреймворк для построения нейронных сетей.
   - Модули `torch` и `torch.nn` предоставляют инструменты для работы с тензорами и создания слоев сети.

2. **Torchvision**:
   - Используется для работы с предобработанными датасетами (например, CIFAR-100) и для преобразования изображений (`transforms`).

3. **PyTorch Lightning**:
   - Упрощает процесс обучения модели, организует код, добавляет удобные инструменты для логирования метрик.
   - Устанавливается командой `!pip install pytorch_lightning`.

4. **Модуль random_split**:
   - Используется для разделения датасета на обучающую и валидационную выборки.

5. **Adam Optimizer**:
   - Оптимизатор, который используется для настройки весов нейронной сети во время обучения.

6. **torch.manual_seed(42)**:
   - Устанавливает фиксированное значение для генератора случайных чисел, чтобы обеспечить воспроизводимость результатов.

После импорта библиотек мы готовы приступить к загрузке данных и их обработке.

In [ ]:
# Устанавливаем и импортируем необходимые библиотеки
import torch
import torchvision
import torchvision.transforms as transforms
from torch import nn
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
!pip install pytorch_lightning
import pytorch_lightning as pl

# Убедимся в воспроизводимости решения
torch.manual_seed(42)

## Подготовка данных

В этом разделе мы подготавливаем данные для обучения нейронной сети. Используется датасет **CIFAR-100**, который доступен через библиотеку Torchvision. Процесс подготовки включает несколько этапов:

1. **Преобразования данных (Transforms)**:
   - Используем `transforms.Compose`, чтобы объединить несколько операций.
   - **ToTensor**: Преобразует изображения в формат тензора PyTorch.
   - **Normalize**: Нормализует изображения с использованием среднего значения `0.5` и стандартного отклонения `0.5` для каждого канала (R, G, B).

2. **Загрузка CIFAR-100**:
   - Датасет автоматически загружается и сохраняется в папке `./data`.
   - Данные разделены на две части:
     - **Обучающая выборка** (train): 50,000 изображений.
     - **Тестовая выборка** (test): 10,000 изображений.

3. **Разделение обучающей выборки**:
   - **Обучающая выборка**: 80% от общего числа обучающих данных.
   - **Валидационная выборка**: 20% от общего числа обучающих данных.
   - Для разделения используется метод `random_split`.

4. **DataLoader**:
   - `DataLoader` используется для создания итераторов данных.
   - Настройки:
     - **batch_size=64**: Мини-пакет содержит 64 изображения.
     - **shuffle=True** (только для обучающей выборки): Перемешивает данные для улучшения качества обучения.

Результатом этого этапа является подготовленный набор данных в виде итераторов для обучения, валидации и тестирования.

In [ ]:
# Преобразования для нормализации данных
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Загрузка CIFAR100 из torchvision
dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

# Разделение на обучающую и валидационную выборки
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Создание DataLoader для каждой выборки
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Files already downloaded and verified
Files already downloaded and verified


## Реализация архитектуры AlexNet

На данном этапе мы реализуем архитектуру **AlexNet**, адаптированную для работы с изображениями из датасета **CIFAR-100** (размеры изображений 32x32). Основные компоненты архитектуры:

1. **Сверточные слои (Convolutional Layers)**:
   - Извлекают пространственные признаки из входных изображений.
   - Активация ReLU добавляет нелинейность, что помогает сети лучше справляться с задачей классификации.
   - Используются пять сверточных слоев с увеличивающимся количеством фильтров (64, 192, 384, 256).

2. **Слои пулинга (Pooling Layers)**:
   - Применяется **MaxPooling** для уменьшения размерности данных и выделения наиболее значимых признаков.
   - Размер ядра пулинга: `2x2`, шаг: `2`.

3. **Полносвязанные слои (Fully Connected Layers)**:
   - Используются для окончательной классификации признаков, извлеченных сверточными слоями.
   - Первый полносвязанный слой принимает вход размером `256 * 4 * 4` (после сверток и пулинга).
   - Второй и третий полносвязанные слои имеют 4096 нейронов.

4. **Dropout**:
   - Применяется после каждого полносвязанного слоя для уменьшения переобучения.

5. **Итоговый слой (Output Layer)**:
   - Линейный слой с числом выходов, равным количеству классов (100), соответствует задаче классификации.

6. **Метод `forward`**:
   - Определяет порядок вычислений:
     - Изображение проходит через сверточные слои (`features`).
     - Полученные признаки преобразуются в одномерный вектор с помощью `torch.flatten`.
     - Вектор подается на вход классификатору (`classifier`), который возвращает выходные вероятности классов.

In [ ]:
# Реализация архитектуры AlexNet
class AlexNet(nn.Module):
    def __init__(self, num_classes=100):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## Настройка колбэков для логирования и сохранения модели

### 1. Колбэк для вывода метрик (PrintMetricsCallback)
Этот колбэк позволяет выводить метрики обучения и валидации после каждой эпохи.
Цель — предоставить ясное представление о процессе обучения.

#### Как работает `PrintMetricsCallback`:
- Метод `on_train_epoch_end` вызывается после завершения каждой эпохи.
- Метрики, такие как `train_loss`, `train_acc`, `val_loss`, и `val_acc`, извлекаются из `trainer.callback_metrics`.
- Метрики выводятся в консоль в следующем формате:
  - `Train Loss`: Потери на обучающей выборке.
  - `Train Accuracy`: Точность на обучающей выборке.
  - `Validation Loss`: Потери на валидационной выборке.
  - `Validation Accuracy`: Точность на валидационной выборке.
- Если метрика недоступна, отображается `N/A`.

Этот колбэк полезен для мониторинга прогресса модели в реальном времени.

---

### 2. Колбэк для сохранения модели (ModelCheckpoint)
`ModelCheckpoint` автоматически сохраняет контрольные точки модели (weights, optimizer state и другие параметры) во время обучения.

#### Как работает `ModelCheckpoint`:
- **`dirpath="checkpoints/"`**: Указывает директорию для сохранения файлов контрольных точек.
- **`filename="alexnet-{epoch:02d}-{val_loss:.2f}"`**: Имя файла содержит номер эпохи и значение `val_loss`.
- **`save_top_k=1`**: Сохраняется только одна лучшая модель с минимальными потерями на валидации.
- **`monitor="val_loss"`**: Указывает, что критерий для сохранения модели — минимизация `val_loss`.
- **`mode="min"`**: Указывает, что модель сохраняется при уменьшении `val_loss`.

Использование этого колбэка гарантирует, что вы сохраните лучшую модель на основе метрик валидации, что особенно полезно для тестирования и развертывания модели.

---

### Результат
- `PrintMetricsCallback` обеспечивает вывод ключевых метрик в консоль после каждой эпохи, улучшая видимость процесса обучения.
- `ModelCheckpoint` позволяет сохранять лучшую версию модели, обеспечивая возможность восстановления обучения или тестирования в любой момент.

In [ ]:
from pytorch_lightning.callbacks import Callback

class PrintMetricsCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get("train_loss_epoch", None)
        train_acc = trainer.callback_metrics.get("train_acc_epoch", None)
        val_loss = trainer.callback_metrics.get("val_loss", None)
        val_acc = trainer.callback_metrics.get("val_acc", None)

        print(f"Epoch {trainer.current_epoch + 1}:")
        print(f"  Train Loss: {train_loss:.4f}" if train_loss else "  Train Loss: N/A")
        print(f"  Train Accuracy: {train_acc:.4f}" if train_acc else "  Train Accuracy: N/A")
        print(f"  Validation Loss: {val_loss:.4f}" if val_loss else "  Validation Loss: N/A")
        print(f"  Validation Accuracy: {val_acc:.4f}" if val_acc else "  Validation Accuracy: N/A")

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="alexnet-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    monitor="val_loss",
    mode="min"
)

## Реализация класса AlexNetLightning

Этот класс реализует модель AlexNet с использованием PyTorch Lightning, что упрощает процесс обучения, валидации и тестирования.

---

### Основные компоненты класса AlexNetLightning

#### 1. Инициализация (`__init__`)
- **Архитектура AlexNet**:
  - Реализована в классе `AlexNet` и адаптирована для работы с CIFAR-100.
- **Функция потерь**:
  - Используется `CrossEntropyLoss`, которая подходит для многоклассовой классификации.
- **Метрики для обучения и оценки**:
  - `Accuracy`: Точность предсказаний (правильных классификаций).
  - `MulticlassPrecision`: Доля правильно предсказанных классов среди всех предсказанных (Precision).
  - `MulticlassRecall`: Доля правильно предсказанных классов среди всех истинных (Recall).
  - `MulticlassF1Score`: Гармоническое среднее Precision и Recall.
- **Списки для сохранения метрик**:
  - `train_losses`, `val_losses`: Сохраняют значения функции потерь для обучения и валидации.
  - `train_accuracies`, `val_accuracies`: Сохраняют значения точности.
  - `test_losses`, `test_accuracies`, `test_precisions`, `test_recalls`, `test_f1_scores`: Хранят метрики, рассчитанные на тестовых данных.

---

### 2. Методы

#### `forward(x)`
- Этот метод определяет, как данные проходят через модель.
- **Вход**: Тензор изображения (например, размером `[batch_size, 3, 32, 32]`).
- **Процесс**: Данные проходят через сверточные и полносвязанные слои AlexNet.
- **Выход**: Вероятности для каждого из 100 классов (размер `[batch_size, 100]`).

#### `training_step(batch, batch_idx)`
- **Цель**: Выполняет один шаг обучения.
- **Параметры**:
  - `batch`: Содержит данные (изображения и соответствующие метки).
  - `batch_idx`: Индекс текущего батча.
- **Процесс**:
  - Прямой проход: данные проходят через модель.
  - Вычисление потерь с помощью функции `CrossEntropyLoss`.
  - Расчет точности (`train_accuracy`) на текущем батче.
- **Логирование**:
  - `train_loss`: Потери для текущего батча.
  - `train_acc`: Точность на текущем батче.
- **Возврат**: Значение потерь, которое используется для обновления весов.

#### `validation_step(batch, batch_idx)`
- **Цель**: Выполняет один шаг валидации.
- **Параметры**:
  - `batch`: Содержит данные (изображения и метки).
  - `batch_idx`: Индекс текущего батча.
- **Процесс**:
  - Прямой проход через модель.
  - Вычисление потерь и точности (`val_accuracy`) для валидационных данных.
- **Логирование**:
  - `val_loss`: Потери на валидации.
  - `val_acc`: Точность на валидационных данных.

#### `test_step(batch, batch_idx)`
- **Цель**: Выполняет один шаг тестирования.
- **Параметры**:
  - `batch`: Содержит данные (изображения и метки).
  - `batch_idx`: Индекс текущего батча.
- **Процесс**:
  - Прямой проход через модель.
  - Вычисление потерь (`test_loss`), точности (`test_accuracy`), Precision, Recall и F1 Score.
  - Каждая метрика сохраняется в соответствующем списке.
- **Логирование**:
  - `test_loss`, `test_acc`, `test_precision`, `test_recall`, `test_f1`.

#### `configure_optimizers()`
- **Цель**: Определяет оптимизатор, используемый для обновления весов модели.
- **Процесс**:
  - Создает объект `Adam` с параметрами модели.
  - Возвращает оптимизатор для использования в процессе обучения.

---

### Результат
1. **Обучение**:
   - Выполняется расчет потерь и точности на каждом батче.
   - Логирование метрик позволяет отслеживать прогресс обучения.
2. **Валидация**:
   - Оценивается производительность модели на валидационных данных.
3. **Тестирование**:
   - Рассчитываются и сохраняются метрики производительности на тестовых данных.
4. **Сохранение метрик**:
   - Метрики записываются в списки, чтобы их можно было использовать для визуализации и анализа после завершения обучения.

In [ ]:
from torchmetrics.classification import Accuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score

class AlexNetLightning(pl.LightningModule):
    def __init__(self, num_classes=100):
        super(AlexNetLightning, self).__init__()
        self.model = AlexNet(num_classes)
        self.criterion = nn.CrossEntropyLoss()

        # Метрики для обучения, валидации и тестирования
        self.train_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_precision = MulticlassPrecision(num_classes=num_classes, average="macro")
        self.test_recall = MulticlassRecall(num_classes=num_classes, average="macro")
        self.test_f1 = MulticlassF1Score(num_classes=num_classes, average="macro")

        # Списки для сохранения метрик
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.test_losses = []
        self.test_accuracies = []
        self.test_precisions = []
        self.test_recalls = []
        self.test_f1_scores = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, targets)
        acc = self.train_accuracy(outputs.softmax(dim=-1), targets)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, targets)
        acc = self.val_accuracy(outputs.softmax(dim=-1), targets)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, targets)
        acc = self.test_accuracy(outputs.softmax(dim=-1), targets)
        precision = self.test_precision(outputs.softmax(dim=-1), targets)
        recall = self.test_recall(outputs.softmax(dim=-1), targets)
        f1 = self.test_f1(outputs.softmax(dim=-1), targets)

        # Сохранение метрик в списки
        self.test_losses.append(loss.item())
        self.test_accuracies.append(acc.item())
        self.test_precisions.append(precision.item())
        self.test_recalls.append(recall.item())
        self.test_f1_scores.append(f1.item())

        self.log('test_loss', loss, on_epoch=True)
        self.log('test_acc', acc, on_epoch=True)
        self.log('test_precision', precision, on_epoch=True)
        self.log('test_recall', recall, on_epoch=True)
        self.log('test_f1', f1, on_epoch=True)

    def configure_optimizers(self):
        return Adam(self.model.parameters())

## Обучение и тестирование модели AlexNet

На этом этапе мы выполняем следующие шаги: инициализация модели, настройка тренера с кастомными колбэками, обучение на 10 эпохах, тестирование модели и вывод ключевых метрик.

---

### Этапы

#### 1. Инициализация модели
- Создается экземпляр класса `AlexNetLightning` с числом выходов, равным числу классов CIFAR-100 (`num_classes=100`).
- Модель готова к обучению, так как уже содержит функции потерь, оптимизатор и метрики.

#### 2. Добавление колбэков
- **PrintMetricsCallback**:
  - Выводит метрики обучения и валидации (Loss, Accuracy) в консоль после каждой эпохи.
  - Удобен для отслеживания прогресса обучения.
- **ModelCheckpoint**:
  - Автоматически сохраняет лучшую версию модели (на основе минимального значения `val_loss`).
  - Обеспечивает возможность восстановления модели для дальнейшего обучения или тестирования.

#### 3. Настройка `Trainer`
- **`max_epochs=10`**:
  - Модель будет обучаться в течение 10 эпох на этом этапе.
- **`devices`**:
  - Устанавливается количество доступных устройств (например, GPU).
- **`accelerator`**:
  - Указывает устройство для обучения: `"gpu"` или `"cpu"`.
- **`callbacks`**:
  - Включает кастомные колбэки для логирования и сохранения модели.

#### 4. Обучение модели
- Метод `trainer.fit` запускает процесс обучения:
  - Использует данные из `train_loader` для обновления весов модели.
  - Выполняет оценку на валидационной выборке (`val_loader`) после каждой эпохи.

#### 5. Тестирование модели
- Метод `trainer.test` оценивает модель на тестовой выборке (`test_loader`):
  - Рассчитывает метрики производительности (`test_loss`, `test_acc`, `test_precision`, `test_recall`, `test_f1`).
  - Все метрики сохраняются в `trainer.callback_metrics`.

#### 6. Вывод тестовых метрик
- Результаты тестирования выводятся в консоль:
  - **`test_loss`**: Средние потери на тестовой выборке.
  - **`test_acc`**: Точность классификации.
  - **`test_precision`**: Точность предсказания.
  - **`test_recall`**: Полнота предсказания.
  - **`test_f1`**: F1 Score — гармоническое среднее Precision и Recall.

---

### Будущие шаги
- После этого этапа можно продолжить обучение модели на большем числе эпох (например, добавить еще 50 эпох).
- Сохраненная лучшая модель (`ModelCheckpoint`) может быть использована для дальнейшего тестирования или развертывания.

In [ ]:
# Инициализация модели
model = AlexNetLightning(num_classes=100)

# Добавляем наш кастомный колбэк
callbacks = [PrintMetricsCallback(), checkpoint_callback]

# Инициализация тренера с колбэками
trainer = pl.Trainer(
    max_epochs=10,
    devices=1 if torch.cuda.is_available() else None,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    callbacks=callbacks
)

# Обучение модели
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /content/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name           | Type                | Params | Mode 
---------------------------------------------------------------
0 | model          | AlexNet             | 36.2 M | train
1 | criterion      | CrossEntropyLoss    | 0      | train
2 | train_accuracy | MulticlassAccuracy  | 0      | train
3 | val_accuracy   | MulticlassAccuracy  | 0      | train
4 | test_accuracy  | MulticlassAccuracy  | 0      | train
5 | test_precision | MulticlassPrecision | 0      | 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 1:
  Train Loss: 4.2176
  Train Accuracy: 0.0421
  Validation Loss: 3.9180
  Validation Accuracy: 0.0723


...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 9:
  Train Loss: 2.4166
  Train Accuracy: 0.3611
  Validation Loss: 2.5391
  Validation Accuracy: 0.3469


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 10:
  Train Loss: 2.3258
  Train Accuracy: 0.3821
  Validation Loss: 2.5101
  Validation Accuracy: 0.3531


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [ ]:
# Тестирование модели
trainer.test(model, test_loader)

# Вывод тестовых метрик
test_results = trainer.callback_metrics
print(f"Test Loss: {test_results['test_loss']:.4f}")
print(f"Test Accuracy: {test_results['test_acc']:.4f}")
print(f"Test Precision: {test_results['test_precision']:.4f}")
print(f"Test Recall: {test_results['test_recall']:.4f}")
print(f"Test F1 Score: {test_results['test_f1']:.4f}")

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.36660000681877136    │
│          test_f1          │    0.2548236846923828     │
│         test_loss         │    2.4976606369018555     │
│      test_precision       │    0.2639555335044861     │
│        test_recall        │    0.27123868465423584    │
└───────────────────────────┴───────────────────────────┘

Test Loss: 2.4977
Test Accuracy: 0.3666
Test Precision: 0.2640
Test Recall: 0.2712
Test F1 Score: 0.2548


## Анализ результатов обучения и тестирования

### Результаты обучения
На основе данных по эпохам:
- **Train Loss (Потери на обучении)**:
  - Потери снижаются с каждой эпохой: от 4.2176 (эпоха 1) до 2.3258 (эпоха 10).
  - Это говорит о том, что модель постепенно улучшает свою способность подгонять обучающие данные.

- **Train Accuracy (Точность на обучении)**:
  - Точность возрастает от 0.0421 (4.21%) на первой эпохе до 0.3821 (38.21%) на последней.
  - Это заметное улучшение, но модель пока далека от идеальной производительности.

- **Validation Loss (Потери на валидации)**:
  - Потери уменьшаются от 3.9180 (эпоха 1) до 2.5101 (эпоха 10), но с меньшей скоростью по сравнению с обучающей выборкой.
  - Это говорит о том, что модель учится, но пока не полностью обобщает информацию на валидационных данных.

- **Validation Accuracy (Точность на валидации)**:
  - Точность растет с 0.0723 (7.23%) до 0.3531 (35.31%) за 10 эпох.
  - Динамика роста точности на валидации соответствует обучающей точности, что указывает на отсутствие серьезного переобучения.

---

### Результаты тестирования
На тестовой выборке:
- **Test Loss**: 2.4977
  - Значение потерь на тестовой выборке близко к потерям на валидации (2.5101), что говорит о согласованности модели.
- **Test Accuracy**: 0.3666 (36.66%)
  - Точность на тестовых данных немного выше, чем на валидации (35.31%), что подтверждает стабильность модели.
- **Test Precision**: 0.2640 (26.40%)
  - Точность предсказаний довольно низкая, что говорит о высоком количестве ложных положительных предсказаний.
- **Test Recall**: 0.2712 (27.12%)
  - Полнота предсказаний также низкая, что указывает на пропуски правильных классов.
- **Test F1 Score**: 0.2548 (25.48%)
  - F1 Score находится между Precision и Recall, отражая общий баланс между этими метриками.

---

### Выводы
1. **Модель учится**, но текущие результаты на тестовой выборке (36.66% точности) показывают, что она еще не достигла своей полной потенциальной производительности.
2. **Метрики Precision, Recall и F1 Score низкие**:
   - Это может указывать на сложности модели с правильным выделением признаков для некоторых классов.
   - Возможно, стоит улучшить качество данных, использовать дополнительные техники регуляризации или доработать архитектуру модели.

---

### Следующие шаги: Обучение на большем количестве эпох
Чтобы улучшить производительность модели:
1. **Продолжение обучения**:
   - Планируется обучить модель на еще **50 эпохах**, чтобы дать ей больше времени на усвоение признаков.
   - Более длительное обучение может улучшить точность и согласованность метрик.

2. **Мониторинг метрик**:
   - Метрики Precision и Recall будут отслеживаться, чтобы оценить способность модели точно предсказывать классы.

3. **Анализ после дополнительных эпох**:
   - Оценим, достигла ли модель насыщения (когда метрики перестают улучшаться) или она продолжает учиться.
   - Если метрики стабилизируются на уровне, ниже ожидаемого, возможно потребуется доработка архитектуры модели или улучшение данных.

## Продолжение обучения модели

В этом разделе мы загружаем ранее сохраненную модель из контрольной точки и продолжаем обучение. Это позволяет использовать уже обученные выше веса.

---

### Этапы

#### 1. Загрузка модели из контрольной точки
- **Контрольная точка**:
  - Файл `alexnet-epoch=09-val_loss=2.51.ckpt` содержит состояние модели после 10 эпох обучения.
  - Он включает:
    - Веса модели.
    - Состояние оптимизатора.
    - Другие параметры, необходимые для продолжения обучения.
- **Команда**:
  - `AlexNetLightning.load_from_checkpoint` загружает контрольную точку и инициализирует модель с теми же параметрами.

#### 2. Настройка нового `Trainer`
- **`max_epochs=60`**:
  - Общее количество эпох обучения увеличено до 60 (10 уже завершено, добавляется еще 50).
- **`devices`**:
  - Настраивается на использование доступного оборудования (GPU или CPU).
- **`callbacks`**:
  - Включает ранее настроенные колбэки:
    - `PrintMetricsCallback` для вывода метрик.
    - `ModelCheckpoint` для сохранения лучшей версии модели.

#### 3. Продолжение обучения
- Метод `trainer.fit` продолжает обучение с использованием загруженной модели и тех же данных (`train_loader` и `val_loader`).
- В процессе:
  - Модель будет дообучена, сохраняя уже полученные знания.

#### 4. Повторное тестирование модели
- После завершения обучения модель снова тестируется на тестовой выборке (`test_loader`) с помощью метода `trainer.test`.
- Тестовые метрики (`test_loss`, `test_acc`, `test_precision`, `test_recall`, `test_f1`) вычисляются и выводятся для анализа.

---

### Результат
- Модель дообучается на большем количестве эпох (до 60), что позволяет улучшить производительность на обучающей и валидационной выборках.
- Повторное тестирование дает обновленные тестовые метрики, позволяя оценить, насколько лучше стала модель после дополнительного обучения.

In [ ]:
# Загрузка модели из контрольной точки
model = AlexNetLightning.load_from_checkpoint("/content/checkpoints/alexnet-epoch=09-val_loss=2.51.ckpt")

# Новый Trainer для продолжения обучения
trainer = pl.Trainer(
    max_epochs=60,
    devices=1 if torch.cuda.is_available() else None,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    callbacks=callbacks
)

# Продолжаем обучение
trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name           | Type                | Params | Mode 
---------------------------------------------------------------
0 | model          | AlexNet             | 36.2 M | train
1 | criterion      | CrossEntropyLoss    | 0      | train
2 | train_accuracy | MulticlassAccuracy  | 0      | train
3 | val_accuracy   | MulticlassAccuracy  | 0      | train
4 | test_accuracy  | MulticlassAccuracy  | 0      

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 1:
  Train Loss: 2.2374
  Train Accuracy: 0.4024
  Validation Loss: 2.4667
  Validation Accuracy: 0.3587


...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 59:
  Train Loss: 0.9104
  Train Accuracy: 0.7394
  Validation Loss: 3.0087
  Validation Accuracy: 0.3743


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=60` reached.


Epoch 60:
  Train Loss: 0.9212
  Train Accuracy: 0.7358
  Validation Loss: 2.9902
  Validation Accuracy: 0.3791


In [ ]:
# Повторное тестирование модели
trainer.test(model, test_loader)

# Вывод тестовых метрик
test_results = trainer.callback_metrics
print(f"Test Loss: {test_results['test_loss']:.4f}")
print(f"Test Accuracy: {test_results['test_acc']:.4f}")
print(f"Test Precision: {test_results['test_precision']:.4f}")
print(f"Test Recall: {test_results['test_recall']:.4f}")
print(f"Test F1 Score: {test_results['test_f1']:.4f}")

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.3912000060081482     │
│          test_f1          │    0.2732997536659241     │
│         test_loss         │    2.9416675567626953     │
│      test_precision       │    0.2849668860435486     │
│        test_recall        │     0.286976158618927     │
└───────────────────────────┴───────────────────────────┘

Test Loss: 2.9417
Test Accuracy: 0.3912
Test Precision: 0.2850
Test Recall: 0.2870
Test F1 Score: 0.2733


## Анализ результатов обучения и тестирования модели

### Результаты обучения на 60 эпохах
На основе данных обучения за 60 эпох можно выделить следующие ключевые моменты:

#### Потери (Loss)
- **Train Loss**:
  - Значение функции потерь на обучающей выборке снизилось с 2.2374 (эпоха 1) до 0.9212 (эпоха 60).
  - Это указывает на то, что модель продолжает хорошо подгонять данные, даже на поздних этапах обучения.
- **Validation Loss**:
  - На валидационной выборке потери уменьшились с 2.4667 (эпоха 1) до 2.9902 (эпоха 60), но после 30-й эпохи началось их повышение.
  - Это может быть признаком переобучения, так как модель начинает хуже обобщать на валидационных данных.

#### Точность (Accuracy)
- **Train Accuracy**:
  - Точность на обучении увеличилась с 0.4024 (40.24%) до 0.7358 (73.58%) за 60 эпох.
  - Это подтверждает, что модель успешно извлекает признаки из обучающих данных.
- **Validation Accuracy**:
  - Точность на валидации возросла с 0.3587 (35.87%) до 0.3791 (37.91%), но прогресс замедлился после 30-й эпохи.
  - Небольшой рост точности в сочетании с увеличением потерь на валидации также указывает на возможное переобучение.

---

### Результаты тестирования
На тестовой выборке модель показала следующие метрики:
- **Test Loss**: 2.9417
  - Потери на тестовой выборке выше, чем на валидационной (2.9902), что указывает на некоторую согласованность между тестовыми и валидационными результатами.
- **Test Accuracy**: 0.3912 (39.12%)
  - Модель улучшила точность по сравнению с предыдущим этапом обучения (36.66%), но прирост остается небольшим.
- **Test Precision**: 0.2850 (28.50%)
  - Точность предсказаний низкая, что может указывать на наличие ложных положительных классификаций.
- **Test Recall**: 0.2870 (28.70%)
  - Полнота также низкая, что говорит о пропусках некоторых правильных классов.
- **Test F1 Score**: 0.2733 (27.33%)
  - Низкий F1 Score подтверждает дисбаланс между Precision и Recall.

---

### Проблемы и выявленные ограничения

#### 1. Переобучение
- Увеличение Validation Loss и замедление роста Validation Accuracy после 30-й эпохи указывает на то, что модель начала запоминать обучающие данные вместо обобщения.

#### 2. Низкие метрики на тестовых данных
- Низкие значения Precision, Recall и F1 Score свидетельствуют о том, что модель затрудняется правильно классифицировать изображения.
- Это может быть вызвано недостаточной сложностью архитектуры модели для задач с 100 классами.

#### 3. Ограниченный прирост точности
- Несмотря на 60 эпох обучения, Test Accuracy увеличилась лишь до 39.12%, что может быть связано с недостаточным выделением признаков или сложностью задачи классификации CIFAR-100.

---

### Пути решения и улучшения результатов

#### 1. Увеличение сложности модели
- **Добавить больше слоев**:
  - Увеличение количества сверточных и полносвязанных слоев может помочь модели извлекать более сложные признаки.
- **Использовать предобученные модели**:
  - Применение более сложных архитектур (например, ResNet, DenseNet) с предварительно обученными весами на ImageNet может существенно улучшить результаты.

#### 2. Регуляризация
- **Dropout**:
  - Увеличение коэффициента Dropout в полносвязанных слоях поможет уменьшить переобучение.
- **L2-регуляризация**:
  - Включение L2-регуляризации в оптимизатор (Weight Decay) уменьшит вероятность переобучения.

#### 3. Data Augmentation
- Добавление разнообразных преобразований изображений (например, случайное вращение, изменение яркости, зеркальное отображение) увеличит разнообразие данных и улучшит обобщающую способность модели.

#### 4. Оптимизация гиперпараметров
- **Скорость обучения (Learning Rate)**:
  - Использование scheduler для динамического изменения скорости обучения.
- **Размер батча (Batch Size)**:
  - Эксперимент с увеличением или уменьшением размера батча для ускорения сходимости.

#### 5. Увеличение объема данных
- Использование дополнительных данных (например, из других датасетов с похожими классами) может помочь модели лучше обучаться и обобщать информацию.

---

### Вывод
Хотя модель показала улучшение метрик на обучающих данных, низкая производительность на тестовой выборке указывает на необходимость внесения улучшений в архитектуру и процесс обучения. Следующие шаги будут направлены на применение вышеуказанных методов для достижения более высоких метрик на тестовых данных.